# Whisper `base` ASR on TIMIT — WER Evaluation

Runs OpenAI Whisper **base** over 10 TIMIT `TRAIN` utterances, prints each model output
against its reference transcript, reports per-file WER, and finishes with summary statistics
(mean WER, median WER, and the count of files with WER >= 1).

## 1. Imports

`whisper` and `jiwer` are already installed in this environment. Uncomment the `pip` lines
if you need to install them.

`soundfile` is used to read the audio: TIMIT `.WAV` files are **NIST SPHERE**, not RIFF/WAVE.
`libsndfile` (behind `soundfile`) reads SPHERE natively and returns a 16 kHz mono float array,
which is exactly what `model.transcribe()` expects.

In [1]:
# !pip install -q openai-whisper jiwer soundfile

import os
import re
import glob
import statistics

import soundfile as sf
import whisper
from jiwer import wer
from IPython.display import Audio, display

## 2. Load the Whisper base model

In [2]:
model = whisper.load_model("base")
print("Loaded Whisper model:", "base")

Loaded Whisper model: base


## 3. Select 10 TIMIT utterances

TIMIT is nested as `TRAIN/<dialect region>/<speaker>/<utterance>.WAV`, so a flat
`os.listdir(TRAIN)` finds **no** `.WAV` files at all — it only sees the `DR1..DR8`
directories. A recursive glob is required.

Selection rules:
- one utterance per speaker, so all 10 files come from 10 different talkers
- round-robin across dialect regions `DR1..DR8`, so the sample is not all one accent
- skip the `SA*` "shibboleth" sentences, which are the identical two sentences recorded
  by every TIMIT speaker and would make the sample repetitive

In [3]:
TIMIT_TRAIN = "/Users/agrfhyl/Documents/timit/TIMIT/TRAIN"
N_FILES = 10

# Recursive glob: TRAIN/DR*/SPEAKER/UTTERANCE.WAV
all_wavs = sorted(glob.glob(os.path.join(TIMIT_TRAIN, "DR*", "*", "*.WAV")))
print(f"Total .WAV files found under TRAIN: {len(all_wavs)}")

# Bucket candidate utterances by dialect region, one per speaker, excluding SA*
by_region = {}
seen_speakers = set()
for path in all_wavs:
    if os.path.basename(path).startswith("SA"):
        continue
    speaker = os.path.basename(os.path.dirname(path))
    if speaker in seen_speakers:
        continue
    seen_speakers.add(speaker)
    region = os.path.basename(os.path.dirname(os.path.dirname(path)))
    by_region.setdefault(region, []).append(path)

# Round-robin across regions until we have N_FILES
audio_files = []
regions = sorted(by_region)
i = 0
while len(audio_files) < N_FILES:
    region = regions[i % len(regions)]
    bucket = by_region[region]
    idx = i // len(regions)
    if idx < len(bucket):
        audio_files.append(bucket[idx])
    i += 1

print(f"Selected {len(audio_files)} files:")
for p in audio_files:
    print("  ", os.path.relpath(p, TIMIT_TRAIN))

Total .WAV files found under TRAIN: 4620
Selected 10 files:
   DR1/FCJF0/SI1027.WAV
   DR2/FAEM0/SI1392.WAV
   DR3/FALK0/SI1086.WAV
   DR4/FALR0/SI1325.WAV
   DR5/FBJL0/SI1552.WAV
   DR6/FAPB0/SI1063.WAV
   DR7/FBLV0/SI1058.WAV
   DR8/FBCG1/SI1612.WAV
   DR1/FDAW0/SI1271.WAV
   DR2/FAJW0/SI1263.WAV


## 4. Reference transcripts and text normalization

Each `X.WAV` has a sibling `X.TXT` holding the reference transcript, formatted as:

```
0 46797 She had your dark suit in greasy wash water all year.
```

The first two integers are start/end sample offsets and must be stripped.

Both reference and hypothesis are normalized before scoring — lowercased, punctuation
removed, hyphens split, whitespace collapsed. Without this, Whisper's casing and
punctuation would count as word errors and inflate WER dramatically.

In [4]:
def load_reference(wav_path):
    """Read the sibling .TXT transcript, dropping the leading sample offsets."""
    txt_path = wav_path[:-4] + ".TXT"
    with open(txt_path, "r") as f:
        # "0 46797 She had your dark suit..." -> "She had your dark suit..."
        return f.read().strip().split(None, 2)[2]


def normalize(text):
    """Lowercase, split hyphens, drop punctuation, collapse whitespace."""
    text = text.lower().replace("-", " ")
    text = re.sub(r"[^a-z' ]", " ", text)
    return " ".join(text.split())


# Sanity check on the first file
_demo = audio_files[0]
print("REF raw :", load_reference(_demo))
print("REF norm:", normalize(load_reference(_demo)))

REF raw : Even then, if she took one step forward he could catch her.
REF norm: even then if she took one step forward he could catch her


## 5. Perform ASR and compute per-file WER

`sf.read(..., dtype="float32")` decodes the NIST SPHERE container to the 16 kHz mono
float32 array Whisper expects, so the array is passed to `transcribe()` directly.

In [5]:
def perform_asr(audio):
    """Transcribe already-decoded 16 kHz mono float32 samples with Whisper."""
    result = model.transcribe(audio, language="en", fp16=False)
    return result["text"].strip()


results = []

for n, audio_file in enumerate(audio_files, start=1):
    # Decode once: TIMIT .WAV files are NIST SPHERE, not RIFF/WAV, so we read the
    # PCM samples with soundfile and reuse that array both for ASR and for playback
    # (IPython.display.Audio can't decode a SPHERE file directly from its bytes).
    audio, sample_rate = sf.read(audio_file, dtype="float32")
    assert sample_rate == 16000, f"expected 16 kHz, got {sample_rate}"

    hypothesis = perform_asr(audio)
    reference = load_reference(audio_file)
    error_rate = wer(normalize(reference), normalize(hypothesis))

    results.append({
        "file": audio_file,
        "reference": reference,
        "hypothesis": hypothesis,
        "wer": error_rate,
    })

    print(f"[{n:2d}/{len(audio_files)}] {os.path.relpath(audio_file, TIMIT_TRAIN)}")
    print(f"     REF: {reference}")
    print(f"     HYP: {hypothesis}")
    print(f"     WER: {error_rate:.4f}")
    display(Audio(data=audio, rate=sample_rate))
    print()

[ 1/10] DR1/FCJF0/SI1027.WAV
     REF: Even then, if she took one step forward he could catch her.
     HYP: even then, if she took one step forward, he could catch her.
     WER: 0.0000


[ 2/10] DR2/FAEM0/SI1392.WAV
     REF: Assume, for example, a situation where a farm has a packing shed and fields.
     HYP: Assume, for example, a situation where a farm has a packing shed and fields.
     WER: 0.0000


[ 3/10] DR3/FALK0/SI1086.WAV
     REF: Then the choreographer must arbitrate.
     HYP: Then the choreographer must arbitrate.
     WER: 0.0000


[ 4/10] DR4/FALR0/SI1325.WAV
     REF: Quite often, honeybees form a majority on the willow catkins.
     HYP: Quite often honeybees form a majority on the willow cackens.
     WER: 0.1000


[ 5/10] DR5/FBJL0/SI1552.WAV
     REF: Outside, only a handful of reporters remained.
     HYP: Outside, only a handful of reporters remained.
     WER: 0.0000


[ 6/10] DR6/FAPB0/SI1063.WAV
     REF: No question ruffles him or causes him to hesitate.
     HYP: Now question ruffles him or causes him to hesitate.
     WER: 0.1111


[ 7/10] DR7/FBLV0/SI1058.WAV
     REF: Readiness exercises are almost continuous.
     HYP: Readiness exercises are almost continuous.
     WER: 0.0000


[ 8/10] DR8/FBCG1/SI1612.WAV
     REF: They've never met, you know.
     HYP: They've never met, you know?
     WER: 0.0000


[ 9/10] DR1/FDAW0/SI1271.WAV
     REF: This has been attributed to helium film flow in the vapor pressure thermometer.
     HYP: This has been attributed to helium film flow in the vapor pressure thermometer.
     WER: 0.0000


[10/10] DR2/FAJW0/SI1263.WAV
     REF: Both have excellent integration of their fiscal tax collection year calendars.
     HYP: Both have excellent integration of the Fiscal Text Collection New Calendars.
     WER: 0.2727


## 6. Summary statistics

In [6]:
wer_values = [r["wer"] for r in results]

average_wer = statistics.mean(wer_values)
median_wer = statistics.median(wer_values)
wer_gte_1_count = sum(1 for w in wer_values if w >= 1)

print(f"Files evaluated:      {len(wer_values)}")
print(f"Average WER:          {average_wer:.4f}")
print(f"Median WER:           {median_wer:.4f}")
print(f"Number of WERs >= 1:  {wer_gte_1_count}")

Files evaluated:      10
Average WER:          0.0484
Median WER:           0.0000
Number of WERs >= 1:  0


### Per-file WER table

In [7]:
print(f"{'File':<28}{'WER':>8}")
print("-" * 36)
for r in sorted(results, key=lambda r: -r["wer"]):
    print(f"{os.path.relpath(r['file'], TIMIT_TRAIN):<28}{r['wer']:>8.4f}")
print("-" * 36)
print(f"{'MEAN':<28}{average_wer:>8.4f}")
print(f"{'MEDIAN':<28}{median_wer:>8.4f}")

File                             WER
------------------------------------
DR2/FAJW0/SI1263.WAV          0.2727
DR6/FAPB0/SI1063.WAV          0.1111
DR4/FALR0/SI1325.WAV          0.1000
DR1/FCJF0/SI1027.WAV          0.0000
DR2/FAEM0/SI1392.WAV          0.0000
DR3/FALK0/SI1086.WAV          0.0000
DR5/FBJL0/SI1552.WAV          0.0000
DR7/FBLV0/SI1058.WAV          0.0000
DR8/FBCG1/SI1612.WAV          0.0000
DR1/FDAW0/SI1271.WAV          0.0000
------------------------------------
MEAN                          0.0484
MEDIAN                        0.0000
